# Full run: Qwen2.5-Omni-7B on CH-SIMS v2 (Kaggle, 2× T4)

Runs everything the proposal needs from this model, in one notebook:

| Task | Clips | What is saved |
| --- | --- | --- |
| `score` | all 4,403 × 4 conditions (T+A+V, T, A, V) | logits of the 7 labels' first tokens + log-sum-exp over the vocabulary |
| `verbalized` | test split (1,036), T+A+V | the model's JSON distribution over the labels (RQ2) |
| `per_modality` | test split (1,036), T+A+V | per-modality labels, conflict flag and distribution (RQ3 prompt) |

Estimated total from the speed test: about 7 GPU-hours (scoring about 3 h; generation is an
estimate, see the proposal). Results are appended line by line to `/kaggle/working/omni_runs/*.jsonl`.

**Running it**
1. *Accelerator*: **GPU T4 x2**; *Internet*: **on**; *Add input*: the preprocessed `chsims-mm` dataset.
2. Use **Save Version → Save & Run All (Commit)** so it runs in the background without the browser open.
3. It stops by itself after `TIME_BUDGET_H` hours. **To continue**, add this notebook's own output
   (*Add input → Your Work → this notebook*) as an input and commit again: finished items are found
   under `/kaggle/input` and skipped, and the new output holds old + new records.
4. For a quick check first, set `LIMIT = 10`.

In [ ]:
# Versions these notebooks were written against (transformers 5.x API: `dtype=`, thinker class).
%pip install -q "transformers==5.20.1" "qwen-omni-utils==0.0.9" audioread librosa

In [ ]:
import glob
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-Omni-7B")  # env override: local tests with a tiny model
LOAD_IN_4BIT = False         # True: 4-bit NF4 weights (needs bitsandbytes), if fp16 does not fit
MAX_MEMORY = {0: "12GiB", 1: "14GiB"}  # leave headroom on GPU 0 for activations
FRAME_MAX_PIXELS = 448 * 252           # per-frame pixel cap passed to the Qwen video processor
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # build inputs only, no model (for local checks)
SEED = 0

# Folder made by analysis/preprocess_chsims.py; found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CHSIMS_MM_DIR")
if DATA_DIR is None:
    hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
    assert hits, "No manifest.csv under /kaggle/input: attach the preprocessed dataset."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

TASKS = ["score", "verbalized", "per_modality"]
GEN_SPLIT = "test"                     # split used for the generation tasks
MAX_NEW_TOKENS = {"verbalized": 96, "per_modality": 320}
TIME_BUDGET_H = 11.0                   # stop starting new items after this many hours
LIMIT = None                           # e.g. 10 for a quick check; None = all clips
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/omni_runs"))
T_START = time.time()

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| out:", OUT_DIR, "| dry run:", DRY_RUN)
print("free disk on / (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), "GiB")

## Clips

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv", dtype={"video_id": str, "clip_id": str})
clips = manifest[manifest["status"] == "ok"].reset_index(drop=True)
if LIMIT:
    clips = pd.concat([clips[clips["mode"] == GEN_SPLIT].head(LIMIT), clips[clips["mode"] != GEN_SPLIT].head(LIMIT)])
gen_clips = clips[clips["mode"] == GEN_SPLIT]
print(len(clips), "clips to score |", len(gen_clips), f"'{GEN_SPLIT}' clips for generation")

## Load the processor and the thinker model

In [ ]:
from qwen_omni_utils import process_mm_info
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
model = None
if not DRY_RUN:
    kwargs = dict(dtype=torch.float16, device_map="auto", max_memory=MAX_MEMORY, attn_implementation="sdpa")
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    t0 = time.time()
    model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
    print(f"loaded in {time.time() - t0:.0f} s")
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} allocated after load: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB")

## Prompts

In [ ]:
LABELS = ["Anger", "Disgust", "Fear", "Joy", "Neutral", "Sadness", "Surprise"]
SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
DESCRIBE = {
    "TAV": "the video frames, the audio track and the transcript",
    "T": "only the transcript",
    "A": "only the audio track",
    "V": "only the video frames, without sound",
}
CONDITIONS = list(DESCRIBE)
ASK = {
    # one label; scored from the logits of its first token
    "label": f"Which emotion does the speaker express? Choose exactly one of: {', '.join(LABELS)}.\nAnswer with the label only.",
    # verbalized distribution
    "distribution": ("Imagine 100 people watched this clip. Estimate how many of them would choose each emotion "
                     f"from: {', '.join(LABELS)}. Reply with JSON only, mapping each label to a count; the counts sum to 100."),
    # RQ3: judge each modality on its own, then aggregate
    "per_modality": (
        "Judge each modality on its own before deciding:\n"
        "- text: what the words alone suggest;\n"
        "- audio: what the tone of voice alone suggests;\n"
        "- video: what the face and body alone suggest.\n"
        "Then say whether the modalities conflict, and estimate how 100 people who watched the whole clip "
        "would split across the emotions.\n"
        f"Use only these labels: {', '.join(LABELS)}. Reply with JSON only, in this form:\n"
        '{"text": {"cue": "<a few words>", "label": "<label>"}, "audio": {"cue": "...", "label": "..."}, '
        '"video": {"cue": "...", "label": "..."}, "conflict": true or false, "distribution": {"<label>": <count>, ...}}'),
}

label_first_ids = [processor.tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(label_first_ids)) == len(LABELS), "two labels share a first token"
# The checkpoint's generation_config sets no end-of-turn token, so without this the model keeps
# writing ("Human: ...") until max_new_tokens.
STOP_IDS = processor.tokenizer.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"])


def build_messages(row, condition, ask="label"):
    content = []
    if "V" in condition:
        frames = sorted(glob.glob(str(DATA_DIR / row["frames_dir"] / "*.jpg")))
        content.append({"type": "video", "video": [f"file://{f}" for f in frames],
                        "sample_fps": float(row["sample_fps"]), "max_pixels": FRAME_MAX_PIXELS})
    if "A" in condition:
        content.append({"type": "audio", "audio": str(DATA_DIR / row["audio_path"])})
    text = f"You are given {DESCRIBE[condition]} of a speaker in a scene from a Chinese film or TV series.\n"
    if "T" in condition:
        text += f"Transcript: {row['text']}\n"
    content.append({"type": "text", "text": text + ASK[ask]})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]


def prepare(messages):
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    audios, images, videos, video_kwargs = process_mm_info(messages, use_audio_in_video=False, return_video_kwargs=True)
    # qwen-omni-utils returns fps as a per-video list; the transformers 5.x processor wants one number.
    fps = video_kwargs.pop("fps", [])
    if fps:
        video_kwargs["fps"] = float(fps[0])
    inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt",
                       padding=True, use_audio_in_video=False, **video_kwargs)
    if model is not None:
        inputs = {k: (v.to(model.device, dtype=model.dtype) if torch.is_floating_point(v) else v.to(model.device))
                  if torch.is_tensor(v) else v for k, v in inputs.items()}
    return inputs


def parse_json(text):
    match = re.search(r"\{.*\}", text, flags=re.S)
    try:
        return json.loads(match.group(0)) if match else None
    except (json.JSONDecodeError, ValueError):
        return None

## Resume

Every record is one JSON line keyed by (model, uid, condition, task). Records from earlier runs,
found anywhere under `/kaggle/input` or already in `OUT_DIR`, are copied into this run's output
and skipped below.

In [ ]:
FILES = {"score": OUT_DIR / "scores.jsonl", "gen": OUT_DIR / "generations.jsonl"}


def key(rec):
    return (rec["model"], rec["uid"], rec["condition"], rec["task"])


done = set()
for kind, path in FILES.items():
    sources = sorted(set(glob.glob(f"/kaggle/input/**/omni_runs/{path.name}", recursive=True)) | ({str(path)} if path.exists() else set()))
    records = {}
    for src in sources:
        with open(src, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    if "error" not in rec:
                        records[key(rec)] = rec
    with open(path, "w", encoding="utf-8") as f:
        for rec in records.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    done |= set(records)
    print(f"{path.name}: {len(records)} finished records from {len(sources)} file(s)")


def save(kind, rec):
    with open(FILES[kind], "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def out_of_time():
    return (time.time() - T_START) / 3600 > TIME_BUDGET_H

## Run

Order: all label scoring first, then the two generation tasks. A clip that fails (for example out of
memory) is saved with an `error` field and retried on the next run.

In [ ]:
@torch.inference_mode()
def run_score(row, condition):
    t0 = time.perf_counter()
    inputs = prepare(build_messages(row, condition, ask="label"))
    t1 = time.perf_counter()
    rec = {"model": MODEL_ID, "uid": row["uid"], "condition": condition, "task": "score",
           "n_tokens": int(inputs["input_ids"].shape[1]), "prep_s": round(t1 - t0, 3)}
    if model is None:
        return rec
    logits = model(**inputs, use_cache=False).logits[0, -1].float()
    rec["gpu_s"] = round(time.perf_counter() - t1, 3)
    rec["finite"] = bool(torch.isfinite(logits).all())
    rec["lse"] = float(torch.logsumexp(logits, dim=-1))  # p(label) = exp(logit - lse)
    rec.update({f"logit_{l}": float(v) for l, v in zip(LABELS, logits[label_first_ids].tolist())})
    return rec


@torch.inference_mode()
def run_generate(row, task):
    ask = "distribution" if task == "verbalized" else task
    inputs = prepare(build_messages(row, "TAV", ask=ask))
    t0 = time.perf_counter()
    rec = {"model": MODEL_ID, "uid": row["uid"], "condition": "TAV", "task": task}
    if model is None:
        return rec
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS[task], do_sample=False,
                         eos_token_id=STOP_IDS, pad_token_id=STOP_IDS[1])
    new = out[0, inputs["input_ids"].shape[1]:]
    rec.update(gen_s=round(time.perf_counter() - t0, 3), n_new_tokens=int(len(new)),
               output=processor.tokenizer.decode(new, skip_special_tokens=True))
    rec["parsed"] = parse_json(rec["output"]) is not None
    return rec


jobs = []
if "score" in TASKS:
    jobs += [("score", row, c) for _, row in clips.iterrows() for c in CONDITIONS]
for task in ("verbalized", "per_modality"):
    if task in TASKS:
        jobs += [(task, row, "TAV") for _, row in gen_clips.iterrows()]
todo = [j for j in jobs if (MODEL_ID, j[1]["uid"], j[2], j[0]) not in done]
print(f"{len(jobs)} items, {len(jobs) - len(todo)} already done, {len(todo)} to run")

if DRY_RUN:  # check that inputs build for a few items of every task, then stop
    for task in TASKS:
        for job in [j for j in todo if j[0] == task][:2]:
            rec = run_score(job[1], job[2]) if task == "score" else run_generate(job[1], task)
            print(task, job[2], rec.get("n_tokens", "inputs ok"))
    todo = []

n_done, stopped_early = 0, False
for task, row, condition in todo:
    if out_of_time():
        stopped_early = True
        break
    try:
        rec = run_score(row, condition) if task == "score" else run_generate(row, task)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        rec = {"model": MODEL_ID, "uid": row["uid"], "condition": condition, "task": task, "error": "oom"}
    save("score" if task == "score" else "gen", rec)
    n_done += 1
    if n_done % 200 == 0:
        print(f"{n_done}/{len(todo)} | {(time.time() - T_START) / 3600:.2f} h elapsed | last task: {task}")

print(f"finished {n_done} items this run", "(stopped at the time budget: commit again to continue)" if stopped_early else "")

## Progress and quick checks

In [ ]:
def read(path):
    return pd.DataFrame([json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]) if path.exists() else pd.DataFrame()


scores, gens = read(FILES["score"]), read(FILES["gen"])
expected = {"score": len(clips) * len(CONDITIONS), "verbalized": len(gen_clips), "per_modality": len(gen_clips)}
for task, n in expected.items():
    df = scores if task == "score" else gens
    got = 0 if df.empty else int(((df["task"] == task) & df.get("error", pd.Series(index=df.index, dtype=object)).isna()).sum())
    print(f"{task:13s} {got:6d} / {n} done")

if not scores.empty and "lse" in scores:
    s = scores.dropna(subset=["lse"])
    mass = np.exp(s[[f"logit_{l}" for l in LABELS]].sub(s["lse"], axis=0)).sum(axis=1)
    print("score: non-finite rows:", int((~s["finite"].astype(bool)).sum()), "| mean label mass by condition:",
          mass.groupby(s["condition"]).mean().round(3).to_dict())
if not gens.empty and "parsed" in gens:
    g = gens.dropna(subset=["parsed"])
    print("generation: JSON parse rate by task:", g.groupby("task")["parsed"].mean().round(3).to_dict(),
          "| mean seconds:", g.groupby("task")["gen_s"].mean().round(2).to_dict())